# 选做内容｜在 Python 中操作 MySQL：从连接到表格分析

适用：第五周3.1.4独立选做；**不纳入正课160分钟，不作为本周必做作业**。
目标：能连接MySQL，用游标查询和增删改，解释commit/rollback，再用pandas读取结果。
先修：变量、列表/元组、函数调用、DataFrame；with和try/except由教师示范，不要求第一次就默写。

业务情境：仓库用MySQL保存产品库存，Python负责读取、更新和分析。下面3种产品、数量、单价均为**人工模拟数据**，不是教材真实工业产量或销售记录。
输入：同目录mysql.local.ini（自行由模板复制并配置）、代码内模拟数据。输出：MySQL教学库里的demo_products_mysql和demo_products_snapshot两张演示表；不写教材CSV或tmp。

验证状态：本版已作Notebook结构、Python语法和离线API核对；**尚未连接真实MySQL执行**。旧SQLite输出已清空，不能作为本版验证证据。见[验证记录](../说明文档/验证记录.md)。

## 学习路线（课外建议约60分钟，环境准备另计）

认识数据库与连接10分钟 → 建表及新增10分钟 → 查询10分钟 → 修改、删除和事务10分钟 → pandas衔接10分钟 → 3个小练习10分钟。

| 名词 | 本案例中的含义 |
|---|---|
| MySQL服务 | 真正保存和处理数据的软件，需要先启动 |
| 数据库 | 教师提供的个人教学库，如python_learning |
| 表/行/列 | 产品表/一个产品/编号、名称、数量、单价 |
| PyMySQL | 让Python发送SQL到MySQL的驱动 |
| connection / cursor | 与数据库的连接 / 在连接中执行语句和提取结果的对象 |
| SQLAlchemy | 本课用于给pandas提供支持MySQL的连接，不学习ORM |

## 0. 教师先准备环境

1. 使用已配置好的MySQL教学服务器（本案例语法以MySQL 8.4官方文档为参照）。Workbench只是客户端，安装它不代表服务器已启动。每位学生使用独立教学库，避免同名表互相影响。
2. 教师在MySQL客户端建立空教学库，例如：

```sql
CREATE DATABASE python_learning CHARACTER SET utf8mb4;
```

教师提供该库的账号，至少具备SELECT、INSERT、UPDATE、DELETE、CREATE权限；不要把生产数据库用于练习。以上SQL在MySQL客户端执行，不是Python单元。
3. 在当前Jupyter所用环境安装依赖；这是供教师按环境执行的命令，本Notebook不会自动安装：

```text
python -m pip install "PyMySQL[rsa]" "SQLAlchemy>=2,<3" pandas
```

`rsa`附加依赖用于MySQL部分认证方式。原课程环境不因本材料自动改变。
4. 将同目录[mysql.example.ini](mysql.example.ini)复制为mysql.local.ini，填写服务器、端口、用户、数据库。密码可留空，在下面通过getpass隐藏输入；不要把真实密码写到Notebook代码或输出。mysql.local.ini已被同目录.gitignore排除，但手工分享文件夹时仍须排除它。
5. 从本周根目录或“选做内容”目录打开本文件，按单元逐步运行。首次完整运行需使用尚无两张演示表的教学库；第二次不要无脑运行全部，见文末“重复运行”。

In [ ]:
from pathlib import Path
from configparser import ConfigParser
from getpass import getpass
from decimal import Decimal
import pymysql
from IPython.display import display

# 可从第五周根目录或本Notebook目录运行。
CONFIG_DIR = Path.cwd()
if not (CONFIG_DIR / 'mysql.example.ini').is_file():
    CONFIG_DIR = CONFIG_DIR / '选做内容'
CONFIG_FILE = CONFIG_DIR / 'mysql.local.ini'
if not CONFIG_FILE.is_file():
    raise FileNotFoundError('请将选做内容/mysql.example.ini复制为mysql.local.ini并填写本地配置。')

## 1. 读取本地配置

host是服务器地址，port通常为3306，database是已经存在的教学库。
关闭INI插值，避免密码中的百分号被当作格式符；不打印配置字典。密码留空时会弹出隐藏输入框。

In [ ]:
config = ConfigParser(interpolation=None)
config.read(CONFIG_FILE, encoding='utf-8-sig')
settings = config['mysql']
password = settings.get('password', '') or getpass('请输入MySQL教学账号密码：')
db_config = {
    'host': settings.get('host', '127.0.0.1'),
    'port': settings.getint('port', 3306),
    'user': settings['user'],
    'password': password,
    'database': settings['database'],
    'charset': 'utf8mb4',
}

## 2. 连接MySQL，先确认连接到哪里

autocommit=False表示修改数据后需要主动提交。DictCursor让每一行以“列名:值”的字典返回，便于初学者看懂。
with conn.cursor()只管理游标，不会自动替你提交事务。执行成功后再学习下一步，不要把连接失败当成空数据。

In [ ]:
conn = pymysql.connect(
    **db_config,
    autocommit=False,
    cursorclass=pymysql.cursors.DictCursor,
    connect_timeout=5,
    read_timeout=10,
    write_timeout=10,
)
with conn.cursor() as cursor:
    cursor.execute('SELECT DATABASE() AS current_database, VERSION() AS mysql_version')
    print(cursor.fetchone())

## 3. 创建一张自己的教学表

id是主键（唯一编号）；VARCHAR存文字；INT存整数；DECIMAL(10,2)存两位小数金额。数量单位为个，单价单位为元/个。ENGINE=InnoDB用于后续事务演示。
建表语句执行一次即可。表已存在会报1050，这样不会误把已有表当空白表重建；不要删除别人的表。
MySQL的CREATE TABLE等DDL可能隐式提交，不能把rollback理解为能撤销所有SQL操作。

In [ ]:
create_sql = """
CREATE TABLE demo_products_mysql (
    id INT PRIMARY KEY,
    product_name VARCHAR(50) NOT NULL,
    quantity INT NOT NULL,
    unit_price DECIMAL(10, 2) NOT NULL
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
"""
with conn.cursor() as cursor:
    cursor.execute(create_sql)
print('教学表已创建。')

## 4. 新增3条模拟产品记录

%s是**数据值的占位符**，整数和文字都使用它，不加引号；值作为另一个参数传给驱动。不能用f-string或字符串拼接填入输入值。
executemany一次处理多组数据。这里用Decimal表达两位小数，避免把金额处理直接建立在二进制小数近似上。
try成功后commit保存，失败时rollback撤销本次未提交修改，再抛出错误让教师检查。

In [ ]:
products = [
    (1, '螺栓', 10, Decimal('2.50')),
    (2, '轴承', 20, Decimal('50.00')),
    (3, '阀门', 5, Decimal('120.00')),
]
insert_sql = 'INSERT INTO demo_products_mysql (id, product_name, quantity, unit_price) VALUES (%s, %s, %s, %s)'
try:
    with conn.cursor() as cursor:
        cursor.executemany(insert_sql, products)
        inserted_count = cursor.rowcount
    conn.commit()
    print('新增记录数：', inserted_count)
except Exception:
    conn.rollback()
    raise

## 5. 查询全部记录

execute发送语句，fetchall取回所有结果，fetchone只取一行。ORDER BY id明确按编号排列，不能依赖数据库默认返回顺序。
预测：结果应有3条记录；名称和数字分别来自哪些字段？

In [ ]:
with conn.cursor() as cursor:
    cursor.execute('SELECT id, product_name, quantity, unit_price FROM demo_products_mysql ORDER BY id')
    rows = cursor.fetchall()
for row in rows:
    print(row)

## 6. 带条件查询

筛选单价至少50元/个的产品。单个参数写成`(min_price,)`，末尾逗号表示这是一个元素的元组。
预测：应返回轴承和阀门。占位符用于**值**，表名、列名不要放入参数，本课表名全部固定。

In [ ]:
min_price = Decimal('50.00')
with conn.cursor() as cursor:
    cursor.execute(
        'SELECT id, product_name, quantity, unit_price FROM demo_products_mysql WHERE unit_price >= %s ORDER BY id',
        (min_price,),
    )
    print(cursor.fetchall())

## 7. 修改指定产品的数量

只把编号2的数量改为25个。WHERE明确修改哪一行；省略它会修改所有行。
commit后，其他新连接也可以读取已提交的结果。rowcount用于观察影响行数；重复设置为相同值时，不必总是1。

In [ ]:
try:
    with conn.cursor() as cursor:
        cursor.execute('UPDATE demo_products_mysql SET quantity = %s WHERE id = %s', (25, 2))
        changed_count = cursor.rowcount
    conn.commit()
    print('修改影响行数：', changed_count)
except Exception:
    conn.rollback()
    raise
with conn.cursor() as cursor:
    cursor.execute('SELECT id, product_name, quantity FROM demo_products_mysql WHERE id = %s', (2,))
    print(cursor.fetchone())

## 8. 演示rollback：撤销未提交的数据修改

临时把螺栓数量改成999，但不提交，立即回滚。随后查询应仍是10。
这只演示InnoDB数据修改事务；已经commit的修改不能靠后面的rollback倒回去，建表也不属于本例可回滚范围。

In [ ]:
try:
    with conn.cursor() as cursor:
        cursor.execute('UPDATE demo_products_mysql SET quantity = %s WHERE id = %s', (999, 1))
finally:
    conn.rollback()
with conn.cursor() as cursor:
    cursor.execute('SELECT id, quantity FROM demo_products_mysql WHERE id = %s', (1,))
    print(cursor.fetchone())

## 9. 删除指定模拟记录

只删除本案例编号3的阀门记录，删除前先查询确认。DELETE删行，DROP TABLE删整张表，两者不是一回事；本Notebook不执行DROP。
成功后剩下编号1和2共2条。删除已提交，关闭Notebook不会自动恢复这条记录。

In [ ]:
try:
    with conn.cursor() as cursor:
        cursor.execute('SELECT id, product_name FROM demo_products_mysql WHERE id = %s', (3,))
        print('本次演示要删除的记录：', cursor.fetchone())
        cursor.execute('DELETE FROM demo_products_mysql WHERE id = %s', (3,))
        deleted_count = cursor.rowcount
    conn.commit()
    print('删除记录数：', deleted_count)
except Exception:
    conn.rollback()
    raise
with conn.cursor() as cursor:
    cursor.execute('SELECT id, product_name, quantity, unit_price FROM demo_products_mysql ORDER BY id')
    print(cursor.fetchall())

## 10. 关闭PyMySQL连接

连接用完应关闭；中途出错或不继续时，也可运行本单元关闭。后续改用SQLAlchemy新建连接，读取前面已提交的数据。

In [ ]:
if 'conn' in globals() and conn.open:
    conn.close()
print('PyMySQL连接已关闭。')

## 11. 用SQLAlchemy为pandas建立MySQL连接（提高）

PyMySQL负责底层连接；SQLAlchemy提供pandas支持的MySQL连接接口。不要把原生PyMySQL连接直接当作pandas推荐的通用连接方式。
URL.create处理含特殊字符的密码，不需要手动拼接连接URL。不打印连接配置。

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from sqlalchemy.types import Integer, String, Numeric

url = URL.create(
    'mysql+pymysql',
    username=db_config['user'],
    password=db_config['password'],
    host=db_config['host'],
    port=db_config['port'],
    database=db_config['database'],
    query={'charset': 'utf8mb4'},
)
engine = create_engine(url, pool_pre_ping=True, connect_args={'connect_timeout': 5})

## 12. pandas读取查询结果，并计算库存金额

对应原教材的read_sql_query/read_sql：MySQL的表清单用SHOW TABLES；筛选结果读成DataFrame后即可沿用3.2的表格操作。
SQLAlchemy text使用`:min_quantity`和参数字典，与直接PyMySQL的`%s`写法不同。
库存金额=数量（个）×单价（元/个），本示例只是库存价值演示，不是收入或利润。

In [ ]:
with engine.connect() as db:
    display(pd.read_sql(text('SHOW TABLES'), con=db))
    frame = pd.read_sql_query(
        text('SELECT id, product_name, quantity, unit_price FROM demo_products_mysql WHERE quantity >= :min_quantity ORDER BY id'),
        con=db,
        params={'min_quantity': 1},
        coerce_float=False,
    )
frame['库存金额/元'] = frame['quantity'] * frame['unit_price']
display(frame)
print('库存金额合计/元：', frame['库存金额/元'].sum())

预期：螺栓10个×2.50元/个=25.00元；轴承25个×50.00元/个=1250.00元；合计1275.00元。设置coerce_float=False，保留MySQL DECIMAL读出的Decimal；这里只显示预期说明，不伪造实际数据库输出。

## 13. 将DataFrame写入一张新表并回读（提高）

对应教材to_sql。只写基础4列，表名固定为demo_products_snapshot，`index=False`不增加DataFrame行号；`if_exists='fail'`在表已存在时停止，避免覆盖已有表。
engine.begin对数据写入成功提交、失败回滚；MySQL建表可能隐式提交，失败后仍须检查是否留下空表，不能承诺DDL也整体回滚。
read_sql_table传的是表名，read_sql_query传的是查询语句；不要像原教材注释那样混淆。

In [ ]:
snapshot = frame[['id', 'product_name', 'quantity', 'unit_price']].copy()
with engine.begin() as db:
    snapshot.to_sql(
        'demo_products_snapshot', con=db, index=False, if_exists='fail',
        dtype={'id': Integer(), 'product_name': String(50), 'quantity': Integer(), 'unit_price': Numeric(10, 2)},
    )
with engine.connect() as db:
    roundtrip = pd.read_sql_table('demo_products_snapshot', con=db, coerce_float=False)
display(roundtrip.sort_values('id'))
print('回读行数：', len(roundtrip))

## 14. 释放连接池

提高部分结束后运行。若中途出错不再继续，也执行本单元释放资源；结束学习后关闭内核，使内存中的账号配置随进程释放。

In [ ]:
if 'engine' in globals():
    engine.dispose()
print('SQLAlchemy连接池已释放。')

## 15. 独立小练习（选做，提交后再反馈）

完成演示后重新运行“连接MySQL”单元。不要重建表、重插入演示数据。以下只是题目，不含已完成答案。

1. 查询数量不少于20个的产品，只返回名称与数量，并口头解释WHERE和参数元组。预期能定位轴承，但必须提交自己的代码。
2. 向教学表新增编号4的“齿轮”，数量8个、单价35.50元/个，提交后再查出来。练习数据也是模拟数据。注意编号重复时不要覆盖已有记录。
3. 将编号4的数量临时改为100个，使用rollback撤销，再查询并解释为什么还是8个。完成后关闭连接。

检查标准：正确连接2分、SQL与绑定参数4分、提交/回滚及关闭连接2分、中文解释2分，共10分，仅供选做自检，不计入本周正课100分。

In [ ]:
# TODO：第1题，重新连接后编写查询

In [ ]:
# TODO：第2题，新增并提交自己的练习记录

In [ ]:
# TODO：第3题，回滚验证并关闭连接

## 重复运行和错误排查

第一次在空教学库中可顺序完成演示。后续只重复查询单元；建表、插入、导出不是可随意反复执行的预览操作。若要完整重做，使用教师另行准备的空教学库；本文件不会自动删表或清空数据。

| 现象 | 排查步骤 |
|---|---|
| ModuleNotFoundError | 核对安装依赖的Python与Notebook内核是否相同 |
| 2003无法连接 | 检查MySQL服务、host、port；安装PyMySQL不会自动安装或启动MySQL |
| 1045拒绝访问 | 本地核对用户名、密码、账号允许的来源主机；不要把密码发到聊天或截图 |
| 1049数据库不存在 | 教师先在服务器创建配置指定的教学库 |
| 1050表已存在 | 可能已经完成建表；转去查询确认，不自动删除已有表 |
| 1062主键重复 | 可能重复运行插入；查询该编号，不用replace掩盖问题 |
| 修改后其他连接看不到 | 检查是否commit；已有连接的事务快照可能仍旧，结束旧事务后重新查询 |
| 密码含@、%等字符 | INI禁用插值；SQLAlchemy用URL.create，不手拼连接字符串 |
| MySQL认证依赖提示 | 核对是否安装PyMySQL[rsa]；由教师检查认证配置 |

## 与原教材3.1.4的对应

原9连接数据库 → 本版1、2及11；原10表清单 → 本版12的SHOW TABLES；原11/12读取SQL结果 → 本版5、6及12；原13写表 → 本版13。新增PyMySQL基础增删改与事务，以便初学者先理解Python如何操作MySQL。

## 官方参考（2026-10-24核对）

- [PyMySQL入门与参数化查询](https://pymysql.readthedocs.io/en/latest/user/examples.html)
- [PyMySQL安装与认证依赖](https://pymysql.readthedocs.io/en/latest/user/installation.html)
- [SQLAlchemy连接与URL.create](https://docs.sqlalchemy.org/en/21/core/engines.html)
- [pandas.read_sql_query](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html)、[DataFrame.to_sql](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html)
- [MySQL事务提交与回滚](https://dev.mysql.com/doc/refman/8.4/en/commit.html)、[DDL隐式提交](https://dev.mysql.com/doc/refman/8.4/en/implicit-commit.html)